# Fréquence améliorée — fichier indépendant du notebook du partenaire

Ce notebook compare des améliorations du modèle de fréquence sans modifier `frequence_regression_logisitique.ipynb`. Tout le code de préparation, d'apprentissage et de prédiction est visible ici. Le résultat attendu est une **probabilité de sinistre**, pas un montant ni une classe binaire.

Nous conservons le découpage commun par client (80 % / 20 %), mais la sélection utilise cinq plis clients dans les 80 %. La validation historique déjà examinée ne décide pas des variantes. La paire fréquence × gravité sera choisie dans `tarification_optimisee.ipynb` sur le RMSE des primes. « Meilleur » signifie meilleur parmi les candidats testés, pas un optimum garanti.

L’analyse exploratoire complète reste disponible dans [frequence_regression_logisitique.ipynb](frequence_regression_logisitique.ipynb). Ce notebook explique les améliorations, leurs comparaisons et le modèle final.

In [1]:
import os
os.environ['OMP_NUM_THREADS']='2'
os.environ['OPENBLAS_NUM_THREADS']='2'
from pathlib import Path
import json,hashlib,time,sys,platform,subprocess,warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cloudpickle,sklearn
from IPython.display import display,Markdown
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler,OneHotEncoder,SplineTransformer,FunctionTransformer
from sklearn.linear_model import LogisticRegression,GammaRegressor
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GroupShuffleSplit,StratifiedGroupKFold,GroupKFold
from sklearn.metrics import log_loss,brier_score_loss,roc_auc_score,root_mean_squared_error,mean_absolute_error,mean_squared_error
from sklearn.exceptions import ConvergenceWarning
from threadpoolctl import threadpool_limits
threadpool_limits(limits=2)
warnings.filterwarnings('error',category=ConvergenceWarning)
racines=[Path.cwd(),Path.cwd()/'TD-Tarification-automobile-par-Machine-Learning']
ROOT=next(p for p in racines if (p/'Data/train.csv').exists())
BASE_OUTPUT=ROOT/'output/optimisation'
BASE_OUTPUT.mkdir(parents=True,exist_ok=True)
SEED=42
train=pd.read_csv(ROOT/'Data/train.csv',dtype={'code_postal':str})
test=pd.read_csv(ROOT/'Data/test.csv',dtype={'code_postal':str})
ii,jj=next(GroupShuffleSplit(n_splits=1,test_size=.2,random_state=SEED).split(train,groups=train.id_client))
dev=train.iloc[ii].copy();valid=train.iloc[jj].copy()
assert set(dev.id_client).isdisjoint(valid.id_client)
assert train['index'].is_unique and test['index'].is_unique
assert set(train.nombre_sinistres.unique())=={0,1}
assert ((train.nombre_sinistres==0)==(train.montant_sinistre==0)).all()
CV_MAIN=StratifiedGroupKFold(n_splits=5,shuffle=True,random_state=20261009)
MAIN_SPLITS=list(CV_MAIN.split(dev,dev.nombre_sinistres,groups=dev.id_client))
DATA_HASH={name:hashlib.sha256((ROOT/'Data'/name).read_bytes()).hexdigest() for name in ['train.csv','test.csv']}
print('Apprentissage commun :',len(dev),'contrats et',int(dev.nombre_sinistres.sum()),'sinistres.')
print('Diagnostic historique réservé :',len(valid),'contrats ; aucune sélection sur ces cibles.')

Apprentissage commun : 40013 contrats et 2352 sinistres.
Diagnostic historique réservé : 9987 contrats ; aucune sélection sur ces cibles.


## 1. Variables et règles métier

Les identifiants, les cibles et les champs de sexe sont exclus des prédicteurs. Les zéros impossibles du poids et de la cylindrée deviennent manquants ; les zéros structurels du second conducteur sont conservés. Le département reste une catégorie, avec les codes 2A/2B et outre-mer gérés comme du texte. Les ratios et interactions utilisent uniquement des informations disponibles avant le sinistre. Aucun taux de sinistre du test n'est calculé.

In [2]:
RAW_FEATURES=['bonus','type_contrat','duree_contrat','anciennete_info','freq_paiement','paiement','utilisation',
    'code_postal','conducteur2','age_conducteur1','age_conducteur2','anciennete_permis1','anciennete_permis2',
    'anciennete_vehicule','cylindre_vehicule','din_vehicule','essence_vehicule','marque_vehicule','modele_vehicule',
    'debut_vente_vehicule','fin_vente_vehicule','vitesse_vehicule','type_vehicule','prix_vehicule','poids_vehicule']
LOW_CATS=['type_contrat','freq_paiement','paiement','utilisation','conducteur2','essence_vehicule','type_vehicule']
NUM_NATIVE=['bonus','duree_contrat','anciennete_info','age_conducteur1','age_conducteur2','anciennete_permis1',
    'anciennete_permis2','anciennete_vehicule','cylindre_vehicule','din_vehicule','prix_vehicule','poids_vehicule',
    'vitesse_vehicule','debut_vente_vehicule','fin_vente_vehicule','puissance_par_tonne']
NUM_COMPACT=['log_bonus','duree_contrat','anciennete_info','age_conducteur1','anciennete_permis2',
    'anciennete_vehicule','log_din','log_poids','log_cylindre']

def preparer_metier(frame):
    needed=['bonus','type_contrat','utilisation','anciennete_vehicule','age_conducteur1',
        'poids_vehicule','din_vehicule','cylindre_vehicule','prix_vehicule','code_postal']
    absent=sorted(set(needed)-set(frame.columns))
    if absent:raise ValueError(f'Colonnes nécessaires absentes : {absent}')
    x=frame.loc[:,[c for c in RAW_FEATURES if c in frame]].copy()
    for col in ['poids_vehicule','cylindre_vehicule']:
        x[col]=x[col].mask(x[col]<=0)
    # Zéros du second conducteur conservés : son absence est une information structurelle.
    s=x.code_postal.astype('string').str.strip().str.upper()
    x['departement']=s.str[:2].where(s.str.fullmatch(r'(?:0[1-9]|1[0-9]|2[1-9]|[3-8][0-9]|9[0-5]|2A|2B)[0-9]{3}',na=False))
    dom=s.str.fullmatch(r'97[1-46][0-9]{2}',na=False)
    x.loc[dom,'departement']=s.loc[dom].str[:3]
    for dest,src in [('log_bonus','bonus'),('log_din','din_vehicule'),('log_poids','poids_vehicule'),
                     ('log_cylindre','cylindre_vehicule'),('log_prix','prix_vehicule')]:
        x[dest]=np.log1p(x[src].mask(x[src]<0))
    x['puissance_par_tonne']=x.din_vehicule/(x.poids_vehicule/1000)
    x['contrat_usage']=x.type_contrat.astype('string').fillna('__NA__')+'|'+x.utilisation.astype('string').fillna('__NA__')
    x['age_maxi']=x.anciennete_vehicule*(x.type_contrat=='Maxi')
    x['prix_maxi']=x.log_prix*(x.type_contrat=='Maxi')
    for col in x.select_dtypes(include=['object','string']).columns:
        x[col]=x[col].astype(object).where(x[col].notna(),np.nan)
    return x

def construire_preparation(numeriques,categorielles,splines=(),min_frequency=50):
    splines=list(splines)
    lineaires=[c for c in numeriques if c not in splines]
    parts=[]
    if lineaires:
        parts.append(('lineaire',Pipeline([('imputation',SimpleImputer(strategy='median',add_indicator=True)),
            ('echelle',StandardScaler())]),lineaires))
    if splines:
        parts.append(('courbes',Pipeline([('imputation',SimpleImputer(strategy='median')),
            ('splines',SplineTransformer(n_knots=4,degree=2,knots='quantile',include_bias=False,extrapolation='constant')),
            ('echelle',StandardScaler())]),splines))
    if categorielles:
        parts.append(('categories',Pipeline([('imputation',SimpleImputer(strategy='constant',fill_value='__NA__')),
            ('encodage',OneHotEncoder(min_frequency=min_frequency,handle_unknown='infrequent_if_exist',sparse_output=False))]),list(categorielles)))
    return ColumnTransformer(parts,remainder='drop')

## 2. Référence : logistique du partenaire avec encodage hors client corrigé

Nous conservons son imputation hiérarchique et son lissage. Dans l'encodage d'apprentissage, les contrats d'un client restent dans le même pli et le prior provient seulement des autres plis. Les données de validation ne participent jamais aux statistiques apprises.

In [3]:
VARIABLES_FREQUENCE=RAW_FEATURES
REDONDANTES = ['debut_vente_vehicule', 'fin_vente_vehicule', 'anciennete_permis1',
               'age_conducteur2', 'prix_vehicule', 'vitesse_vehicule']
A_IMPUTER = ['poids_vehicule', 'cylindre_vehicule', 'anciennete_vehicule']
QUEUE_LOURDE = ['bonus', 'din_vehicule', 'poids_vehicule', 'cylindre_vehicule']
CAT_FAIBLE = ['type_contrat', 'freq_paiement', 'paiement', 'utilisation',
              'conducteur2', 'essence_vehicule', 'type_vehicule']
CAT_FORTE = ['marque_vehicule', 'modele_vehicule', 'departement']
SEUIL_RARE, LISSAGE, C_FREQUENCE = 50, 100, 0.1

def nettoyer_frequence(frame):
    manque = sorted(set(VARIABLES_FREQUENCE) - set(frame.columns))
    if manque:
        raise ValueError(f'Colonnes de fréquence absentes : {manque}')
    # Sélection explicite : les cibles présentes dans train ne peuvent pas entrer.
    x = frame.loc[:, VARIABLES_FREQUENCE].copy()
    for col in ['poids_vehicule', 'cylindre_vehicule']:
        x[col] = x[col].replace(0, np.nan)
    x['departement'] = x['code_postal'].astype('string').str[:2]
    x = x.drop(columns='code_postal')
    for col in CAT_FAIBLE + CAT_FORTE:
        x[col] = x[col].astype(object).where(x[col].notna(), '__MANQUANT__')
    return x

def taux_lisse(x, y, prior):
    stats = y.groupby(x).agg(['sum', 'count'])
    return (stats['sum'] + LISSAGE * prior) / (stats['count'] + LISSAGE)

def encoder_hors_clients(x, y, groupes):
    cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
    encodage = pd.DataFrame(np.nan, index=x.index, columns=CAT_FORTE)
    audit = []
    for k, (a, b) in enumerate(cv.split(x, y, groups=groupes), 1):
        assert set(groupes.iloc[a]).isdisjoint(groupes.iloc[b])
        prior = float(y.iloc[a].mean())
        for col in CAT_FORTE:
            mapping = taux_lisse(x.iloc[a][col], y.iloc[a], prior)
            encodage.loc[x.index[b], col] = x.iloc[b][col].map(mapping).fillna(prior).to_numpy()
        audit.append({'pli': k, 'contrats_apprentissage': len(a),
            'contrats_encodés': len(b), 'clients_communs': 0, 'prior_autres_plis': prior})
    assert np.isfinite(encodage.to_numpy()).all()
    return encodage, pd.DataFrame(audit)

def transformer_frequence(frame, p, encodage=None):
    x = nettoyer_frequence(frame).drop(columns=REDONDANTES)
    for col in A_IMPUTER:
        x[col] = (x[col].fillna(x.modele_vehicule.map(p[f'med_modele_{col}']))
            .fillna(x.marque_vehicule.map(p[f'med_marque_{col}']))
            .fillna(p[f'med_globale_{col}']))
    for col in QUEUE_LOURDE:
        if (x[col] < 0).any():
            raise ValueError(f'Valeur négative incompatible avec log1p : {col}')
        x[col] = np.log1p(x[col])
    for col in CAT_FORTE:
        if encodage is not None:
            assert encodage.index.equals(x.index)
            x[col] = encodage[col]
        else:
            x[col] = x[col].map(p[f'taux_{col}']).fillna(p['taux_moyen'])
    for col in CAT_FAIBLE:
        x[col] = x[col].where(x[col].isin(p[f'frequentes_{col}']), 'Autre')
    x = pd.get_dummies(x, columns=CAT_FAIBLE, dtype=int)
    x = x.drop(columns=[f'{col}_{p[f"reference_{col}"]}' for col in CAT_FAIBLE], errors='ignore')
    if 'colonnes' in p:
        x = x.reindex(columns=p['colonnes'], fill_value=0)
    return x.astype(float)

def standardiser_frequence(x, p):
    x = x.copy()
    cols = p['colonnes_num']
    x[cols] = (x[cols] - p['moyennes']) / p['ecarts_types']
    if not np.isfinite(x.to_numpy()).all():
        raise ValueError('Préparation de fréquence non finie.')
    return x

def apprendre_frequence(frame, y, groupes):
    assert frame.index.is_unique and frame.index.equals(y.index)
    assert frame.index.equals(groupes.index)
    x = nettoyer_frequence(frame)
    p = {'taux_moyen': float(y.mean())}
    for col in A_IMPUTER:
        p[f'med_modele_{col}'] = x.groupby('modele_vehicule')[col].median()
        p[f'med_marque_{col}'] = x.groupby('marque_vehicule')[col].median()
        p[f'med_globale_{col}'] = float(x[col].median())
    for col in CAT_FAIBLE:
        counts = x[col].value_counts()
        p[f'frequentes_{col}'] = counts[counts >= SEUIL_RARE].index
        p[f'reference_{col}'] = counts.index[0]
    for col in CAT_FORTE:
        p[f'taux_{col}'] = taux_lisse(x[col], y, p['taux_moyen'])
    oof, audit = encoder_hors_clients(x, y, groupes)
    z = transformer_frequence(frame, p, encodage=oof)
    p['colonnes'] = z.columns
    p['colonnes_num'] = [col for col in z if z[col].nunique() > 2]
    p['moyennes'] = z[p['colonnes_num']].mean()
    p['ecarts_types'] = z[p['colonnes_num']].std().replace(0, 1)
    return p, standardiser_frequence(z, p), audit

def preparer_frequence(frame, p):
    # Validation / test : aucun y, toutes les statistiques viennent de p.
    return standardiser_frequence(transformer_frequence(frame, p), p)

## 3. Améliorations comparées

- Logistiques avec prix, ratio puissance/poids, courbes par splines des âges et de la durée, interactions contrat × usage, et catégories regroupées.
- Boostings prudents avec variables numériques et catégories natives, réglés sans arrêt anticipé sur un sous-échantillon aléatoire qui mélangerait les clients.
- Une calibration sigmoïde apprise sur des probabilités hors client à l'intérieur de chaque apprentissage.
- Des mélanges à poids fixes de la référence et du boosting : aucun poids n'est choisi sur la validation historique.

Les splines enrichissent les entrées de la logistique ; elles restent apprises dans le pli. Les catégories de voiture rares sont regroupées pour limiter leur cardinalité. Aucun rééquilibrage automatique des classes n'est appliqué : nous cherchons des probabilités pour une prime.

In [4]:

def cache_prediction(X,y,Xeval,config,component,fit_predict):
    signature=hashlib.sha256(CACHE_SIGNATURE.encode())
    signature.update(np.asarray(X['index'],dtype=np.int64).tobytes())
    signature.update(np.asarray(Xeval['index'],dtype=np.int64).tobytes())
    signature.update(np.asarray(y,dtype=np.float64).tobytes())
    signature.update(json.dumps(config,sort_keys=True).encode())
    cache_dir=BASE_OUTPUT/'cache'/component
    cache_dir.mkdir(parents=True,exist_ok=True)
    target=cache_dir/(signature.hexdigest()+'.npy')
    if target.exists():
        result=np.load(target,allow_pickle=False)
        if result.shape==(len(Xeval),) and np.isfinite(result).all():return result
    started=time.perf_counter()
    result=np.asarray(fit_predict(),dtype=float)
    assert result.shape==(len(Xeval),) and np.isfinite(result).all()
    temporary=target.with_suffix('.tmp')
    with temporary.open('wb') as f:np.save(f,result,allow_pickle=False)
    temporary.replace(target)
    print(component,config['name'],':',round(time.perf_counter()-started,1),'s',flush=True)
    return result

class ReferenceFrequence:
    def fit(self,X,y,groups):
        y=pd.Series(np.asarray(y),index=X.index)
        groups=pd.Series(np.asarray(groups),index=X.index)
        self.params_,z,self.audit_=apprendre_frequence(X,y,groups)
        self.model_=LogisticRegression(C=.1,max_iter=3000)
        self.model_.fit(z,y)
        return self
    def predict_proba(self,X):return self.model_.predict_proba(preparer_frequence(X,self.params_))

class LogistiqueMetier:
    def __init__(self,config):self.config=config
    def fit(self,X,y,groups=None):
        c=self.config
        nums=NUM_COMPACT.copy()
        cats=LOW_CATS.copy()
        if c.get('prix'):nums+=['log_prix','puissance_par_tonne']
        if c.get('interaction'):nums+=['age_maxi','prix_maxi'];cats+=['contrat_usage']
        if c.get('geographie'):cats+=['departement']
        if c.get('vehicules'):cats+=['marque_vehicule','modele_vehicule']
        self.prep_=construire_preparation(nums,cats,['anciennete_vehicule','age_conducteur1','duree_contrat'] if c.get('splines') else [],c.get('rare',50))
        z=self.prep_.fit_transform(preparer_metier(X))
        self.model_=LogisticRegression(C=c['C'],max_iter=4000,tol=1e-5)
        self.model_.fit(z,y)
        return self
    def predict_proba(self,X):return self.model_.predict_proba(self.prep_.transform(preparer_metier(X)))

class BoostingMetier:
    def __init__(self,config):self.config=config
    def _frame(self,X,fit=False):
        x=preparer_metier(X)
        cats=LOW_CATS+['departement','marque_vehicule','modele_vehicule','contrat_usage']
        x=x.loc[:,NUM_NATIVE+cats].copy()
        if fit:self.categories_={}
        for col in cats:
            s=x[col].fillna('__NA__').astype(str)
            if fit:
                vc=s.value_counts()
                self.categories_[col]=list(vc[vc>=self.config.get('rare',100)].index[:200])
            keep=self.categories_[col]
            s=s.where(s.isin(keep),'__RARE__')
            x[col]=pd.Categorical(s,categories=keep+['__RARE__'])
        return x
    def fit(self,X,y,groups=None):
        c=self.config
        self.model_=HistGradientBoostingClassifier(learning_rate=c['lr'],max_leaf_nodes=c['leaves'],
            max_iter=c['iterations'],min_samples_leaf=c['min_leaf'],l2_regularization=c['l2'],
            categorical_features='from_dtype',early_stopping=False,random_state=42)
        self.model_.fit(self._frame(X,fit=True),y)
        return self
    def predict_proba(self,X):return self.model_.predict_proba(self._frame(X))

def logit_safe(p):
    p=np.clip(p,1e-6,1-1e-6)
    return np.log(p/(1-p)).reshape(-1,1)

class FrequenceCalibree:
    def __init__(self,config):self.config=config
    def fit(self,X,y,groups):
        y=np.asarray(y);groups=np.asarray(groups)
        oof=np.full(len(X),np.nan)
        cv=StratifiedGroupKFold(n_splits=3,shuffle=True,random_state=1803)
        for a,b in cv.split(X,y,groups=groups):
            assert set(groups[a]).isdisjoint(groups[b])
            m=BoostingMetier(self.config['base']).fit(X.iloc[a],y[a],groups[a])
            oof[b]=m.predict_proba(X.iloc[b])[:,1]
        assert np.isfinite(oof).all()
        self.calibration_=LogisticRegression(C=10,max_iter=1000)
        self.calibration_.fit(logit_safe(oof),y)
        self.base_=BoostingMetier(self.config['base']).fit(X,y,groups)
        return self
    def predict_proba(self,X):return self.calibration_.predict_proba(logit_safe(self.base_.predict_proba(X)[:,1]))

class MelangeFrequence:
    def __init__(self,config):self.config=config
    def fit(self,X,y,groups):
        self.models_=[creer_frequence(c).fit(X,y,groups) for c in self.config['membres']]
        return self
    def predict_proba(self,X):
        w=self.config['poids']
        p=w*self.models_[0].predict_proba(X)[:,1]+(1-w)*self.models_[1].predict_proba(X)[:,1]
        return np.column_stack([1-p,p])

def creer_frequence(config):
    return {'reference':ReferenceFrequence,'logistique':LogistiqueMetier,
        'boosting':BoostingMetier,'calibration':FrequenceCalibree,'melange':MelangeFrequence}[config['kind']](config) if config['kind']!='reference' else ReferenceFrequence()

FREQUENCES=[{'name':'Logistique_reference','kind':'reference'}]
for label,kwargs in [
    ('lineaire_prix',{'prix':True}),
    ('splines_prix',{'prix':True,'splines':True}),
    ('splines_interactions',{'prix':True,'splines':True,'interaction':True}),
    ('splines_geo_vehicules',{'prix':True,'splines':True,'geographie':True,'vehicules':True}),
]:
    for C in [.1,1.]:FREQUENCES.append({'name':f'Logistique_{label}_C{C}','kind':'logistique','C':C,**kwargs})
for label,leaves,it,minleaf,l2 in [
    ('prudent',3,200,300,5),('souple',7,250,200,5),
    ('regulier',7,350,300,20),('interactions',15,200,300,20)]:
    FREQUENCES.append({'name':f'Boosting_{label}','kind':'boosting','leaves':leaves,'iterations':it,
        'min_leaf':minleaf,'l2':l2,'lr':.05,'rare':100})
boost_base=next(c for c in FREQUENCES if c['name']=='Boosting_souple')
FREQUENCES.append({'name':'Boosting_souple_calibre','kind':'calibration','base':boost_base})
reference=FREQUENCES[0]
for w in [.25,.5,.75]:
    FREQUENCES.append({'name':f'Melange_logistique_boosting_{w}','kind':'melange',
        'membres':[reference,boost_base],'poids':w})

def ajuster_frequences(X,y,groups,Xeval,configs):
    predictions={}
    for c in configs:
        if c['kind']=='melange':
            w=c['poids'];m1,m2=c['membres']
            predictions[c['name']]=w*predictions[m1['name']]+(1-w)*predictions[m2['name']]
        else:
            predictions[c['name']]=cache_prediction(X,y,Xeval,c,'frequence',
                lambda c=c: creer_frequence(c).fit(X,y,groups).predict_proba(Xeval)[:,1])
    result=np.column_stack([predictions[c['name']] for c in configs])
    assert np.isfinite(result).all() and ((result>=0)&(result<=1)).all()
    return result
print(len(FREQUENCES),'candidats fréquence, dont des mélanges fixes sans sélection de poids sur la validation.')

17 candidats fréquence, dont des mélanges fixes sans sélection de poids sur la validation.


## 4. Validation croisée commune

Chaque candidat prédit chaque contrat de développement une fois hors apprentissage. Imputations, catégories, splines, encodages de cible et calibration sont réappris uniquement dans le pli. Le tableau log-loss/Brier/AUC sert au diagnostic de fréquence ; le choix de tarification utilise le produit avec Gamma. Les prédictions hors pli sont exportées avec leurs identifiants pour vérifier l'alignement.

Les prédictions intermédiaires sont mises en cache avec une empreinte du code, des données, des identifiants, des cibles d’apprentissage et des paramètres. Une reprise ne réutilise que des calculs identiques ; aucun score historique ne sert à décider quels plis conserver.

In [5]:
import nbformat
_source_nb=nbformat.read(ROOT/'modele_frequence_ameliore.ipynb',as_version=4)
_source_code='\n'.join(c.source for c in _source_nb.cells[:9] if c.cell_type=='code')
CACHE_SIGNATURE=hashlib.sha256((_source_code+json.dumps(DATA_HASH,sort_keys=True)+sklearn.__version__).encode()).hexdigest()
OUT=BASE_OUTPUT/'frequence';OUT.mkdir(parents=True,exist_ok=True)
names=[c['name'] for c in FREQUENCES]
oof=np.full((len(dev),len(names)),np.nan);fold_rows=[]
started=time.perf_counter()
for fold,(a,b) in enumerate(MAIN_SPLITS,1):
    assert set(dev.iloc[a].id_client).isdisjoint(dev.iloc[b].id_client)
    p=ajuster_frequences(dev.iloc[a],dev.iloc[a].nombre_sinistres,dev.iloc[a].id_client,dev.iloc[b],FREQUENCES)
    oof[b]=p
    for j,name in enumerate(names):
        fold_rows.append({'pli':fold,'modele':name,'log_loss':log_loss(dev.iloc[b].nombre_sinistres,p[:,j]),
            'Brier':brier_score_loss(dev.iloc[b].nombre_sinistres,p[:,j]),'AUC':roc_auc_score(dev.iloc[b].nombre_sinistres,p[:,j])})
    (BASE_OUTPUT/'progression.json').write_text(json.dumps({'etape':'CV frequence','pli':fold,'plis':5,'secondes':time.perf_counter()-started}))
    print('Fréquence : pli',fold,'/ 5 terminé.',flush=True)
assert np.isfinite(oof).all()
np.savez_compressed(OUT/'predictions_oof.npz',ids=dev['index'].to_numpy(),predictions=oof,names=np.array(names))
pd.DataFrame(fold_rows).to_csv(OUT/'scores_plis.csv',index=False)
table=pd.DataFrame([{'modele':name,'log_loss_OOF':log_loss(dev.nombre_sinistres,oof[:,j]),
    'Brier_OOF':brier_score_loss(dev.nombre_sinistres,oof[:,j]),'AUC_OOF':roc_auc_score(dev.nombre_sinistres,oof[:,j]),
    'ratio_sinistres_predits_observes':oof[:,j].sum()/dev.nombre_sinistres.sum()}
    for j,name in enumerate(names)]).sort_values('log_loss_OOF')
table.to_csv(OUT/'comparaison_cv.csv',index=False)
(OUT/'catalogue.json').write_text(json.dumps({'configs':FREQUENCES,'sha256_data':DATA_HASH},ensure_ascii=False,indent=2))
with (OUT/'fabrique.pkl').open('wb') as f:cloudpickle.dump({'make':creer_frequence,'batch':ajuster_frequences,'configs':FREQUENCES},f)
display(table)
print('Meilleure fréquence en log-loss seule :',table.iloc[0].modele)
print('La paire finale sera choisie sur la RMSE des primes, pas sur cette seule colonne.')

frequence Logistique_reference : 17.9 s
frequence Logistique_lineaire_prix_C0.1 : 0.7 s
frequence Logistique_lineaire_prix_C1.0 : 1.2 s
frequence Logistique_splines_prix_C0.1 : 1.0 s
frequence Logistique_splines_prix_C1.0 : 1.2 s
frequence Logistique_splines_interactions_C0.1 : 0.8 s
frequence Logistique_splines_interactions_C1.0 : 1.4 s
frequence Logistique_splines_geo_vehicules_C0.1 : 2.3 s
frequence Logistique_splines_geo_vehicules_C1.0 : 5.3 s
frequence Boosting_prudent : 1.5 s
frequence Boosting_souple : 2.2 s
frequence Boosting_regulier : 3.5 s
frequence Boosting_interactions : 2.0 s
frequence Boosting_souple_calibre : 12.2 s
Fréquence : pli 1 / 5 terminé.
frequence Logistique_reference : 10.4 s
frequence Logistique_lineaire_prix_C0.1 : 1.0 s
frequence Logistique_lineaire_prix_C1.0 : 1.3 s
frequence Logistique_splines_prix_C0.1 : 1.4 s
frequence Logistique_splines_prix_C1.0 : 1.9 s
frequence Logistique_splines_interactions_C0.1 : 1.4 s
frequence Logistique_splines_interactions_C1

,modele,log_loss_OOF,Brier_OOF,AUC_OOF,ratio_sinistres_predits_observes
16,Melange_logistique_boosting_0.75,0.215726,0.054541,0.645093,0.998921
1,Logistique_lineaire_prix_C0.1,0.215737,0.054544,0.645541,1.000169
5,Logistique_splines_interactions_C0.1,0.215740,0.054539,0.645813,1.000015
6,Logistique_splines_interactions_C1.0,0.215758,0.054542,0.646160,1.000011
4,Logistique_splines_prix_C1.0,0.215762,0.054549,0.645150,1.000037
0,Logistique_reference,0.215779,0.054547,0.645623,1.000713
3,Logistique_splines_prix_C0.1,0.215784,0.054547,0.644727,1.000087
2,Logistique_lineaire_prix_C1.0,0.215785,0.054551,0.645240,1.000152
15,Melange_logistique_boosting_0.5,0.216344,0.054632,0.640159,0.997129
7,Logistique_splines_geo_vehicules_C0.1,0.216469,0.054619,0.639104,1.000242


## 5. Modèle final et utilisation

Après l'assemblage, la cellule suivante affiche le modèle retenu et recharge sa version réentraînée sur tous les contrats du train. Le notebook initial du partenaire reste indépendant de ces expériences.

### Comprendre le modèle retenu

Une régression logistique calcule une somme de coefficients puis la transforme en probabilité avec la fonction sigmoïde. Le boosting ajoute des arbres peu profonds, qui peuvent représenter des seuils et des interactions. Lorsqu'un mélange est retenu, sa probabilité est la moyenne pondérée des probabilités de ses membres, avec le poids indiqué dans la configuration finale.

L'apprentissage utilise tous les contrats du train et la cible `nombre_sinistres`. La validation croisée réserve des **clients entiers**, et les encodages de cible sont eux-mêmes appris hors client. Le CSV test ne contient aucune cible : il sert à prédire après le choix et le réentraînement final. Les métriques de classification évaluent la fréquence ; le choix final de la paire porte sur la RMSE de prime. Le notebook d'origine du partenaire reste inchangé.

In [6]:
COMPONENT='frequence'
PICKLE_NAME='modele_frequence_final.pkl'

config_path=BASE_OUTPUT/'prime/configuration_finale.json'
if config_path.exists():
    final_config=json.loads(config_path.read_text())
    choix=final_config[COMPONENT]
    display(Markdown('## MODÈLE FINAL RETENU POUR LA PRIME\n\n**'+choix['name']+'**\n\nParamètres : `'+json.dumps(choix,ensure_ascii=False)+'`'))
    with (BASE_OUTPUT/'prime'/PICKLE_NAME).open('rb') as f:modele_final=cloudpickle.load(f)
    sorties=modele_final.predict_proba(test.iloc[:5])[:,1] if COMPONENT=='frequence' else modele_final.predict(test.iloc[:5])
    display(pd.DataFrame({'index':test['index'].iloc[:5].to_numpy(),'prediction_composante':sorties}))

    if choix.get('kind')=='melange':
        display(Markdown(f"Probabilité finale = **{choix['poids']:.0%}** × probabilité de `{choix['membres'][0]['name']}` + **{1-choix['poids']:.0%}** × probabilité de `{choix['membres'][1]['name']}`."))
    table_finale=pd.read_csv(BASE_OUTPUT/'frequence/comparaison_cv.csv')
    display(table_finale.loc[table_finale.modele.isin([choix['name'],'Logistique_reference'])])
    print('Les résultats OOF ont participé à la sélection : le diagnostic de gain est dans tarification_optimisee.ipynb.')
else:
    print('Catalogue prêt. Exécuter tarification_optimisee.ipynb pour choisir la paire et entraîner la version finale.')

Les résultats OOF ont participé à la sélection : le diagnostic de gain est dans tarification_optimisee.ipynb.


## MODÈLE FINAL RETENU POUR LA PRIME

**Melange_logistique_boosting_0.75**

Paramètres : `{"name": "Melange_logistique_boosting_0.75", "kind": "melange", "membres": [{"name": "Logistique_reference", "kind": "reference"}, {"name": "Boosting_souple", "kind": "boosting", "leaves": 7, "iterations": 250, "min_leaf": 200, "l2": 5, "lr": 0.05, "rare": 100}], "poids": 0.75}`

,index,prediction_composante
0,50000,0.037575
1,50001,0.027969
2,50002,0.110012
3,50003,0.064332
4,50004,0.024588


Probabilité finale = **75%** × probabilité de `Logistique_reference` + **25%** × probabilité de `Boosting_souple`.

,modele,log_loss_OOF,Brier_OOF,AUC_OOF,ratio_sinistres_predits_observes
0,Melange_logistique_boosting_0.75,0.215726,0.054541,0.645093,0.998921
5,Logistique_reference,0.215779,0.054547,0.645623,1.000713


Références : [SplineTransformer](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.SplineTransformer.html), [HistGradientBoostingClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.HistGradientBoostingClassifier.html). Les gains doivent être jugés avec la validation imbriquée de la paire, et pas seulement avec le meilleur score de ce tableau.